# EYES-DEFY-ANEMIA -- Fine-tune Pilot -- CoAtNet-3

Partial fine-tuning: unfreeze attention sub-layer of stage 4's last block (`stages[3].blocks[1]`, 9.46M params) + head of CoAtNet-3, continuing training
from the already-converged frozen-backbone checkpoint (`Output/version1/checkpoints/best_coatnet_3_palpebral_new_way.pth`,
val F1=0.8966) rather than starting from a fresh head.

Full design rationale (why this submodule, discriminative LRs, the BatchNorm check) is in
`classification/new_way/Fine_tune/finetune_engine_coatnet3.py`'s module docstring and
`classification/.project_memory/13_finetune_pilot_programme.md`.

**Already run locally** (RTX 4050) as a fast pilot -- best val F1 reached there was
**0.8571**, below the frozen-backbone baseline. This notebook reproduces that same,
already-settled configuration on Kaggle for the citable/official record, matching this project's
convention that real training results live on Kaggle.

**Data:** TRAIN reads the offline-balanced + online-augmented data (`Offline_data_augmentation/`,
arrives via `git clone` -- small enough to be committed directly, no separate dataset needed for
it). VAL/TEST read real, unmodified images from the `processed-dataset-clean` Kaggle dataset
(same one every other `new_way/` notebook uses). **The source checkpoint above is NOT in git**
(gitignored, large binary) -- it must be attached as its own Kaggle dataset; see the Data section
below.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first.
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 2376, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 2376 (delta 22), reused 73 (delta 21), pack-reused 2301 (from 1)
Receiving objects: 100% (2376/2376), 104.64 MiB | 33.36 MiB/s, done.
Resolving deltas: 100% (832/832), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic only -- run this BEFORE filling in the TODO paths in the Data section below.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [4]:
# optuna is required even though this notebook never runs a search -- datapreparepipeline/
# trainer_engine.py (reused for compute_metrics/evaluate) imports it unconditionally at module level.
# timm is required -- CoAtNet-3 does not exist in torchvision at all.
!pip install -q optuna albumentations timm

## Data

Two things need to be attached as Kaggle datasets for this notebook to run:

1. **`processed-dataset-clean`** (same dataset every other `new_way/` notebook uses) -- provides the
   real VAL/TEST images plus `splits.csv`/`extraction_log.csv`.
2. **A new dataset containing the 3 original `new_way` checkpoints** (`Output/version1/checkpoints/`
   is `.gitignore`'d -- these were never pushed to GitHub). Zip and upload
   `best_convnext_base_palpebral_new_way.pth`, `best_coatnet_3_palpebral_new_way.pth`, and
   `best_efficientnet_b3_forniceal_palpebral_new_way.pth` together as one Kaggle dataset (~1GB total,
   only best_coatnet_3_palpebral_new_way.pth is actually used by this specific notebook) and attach it here
   too -- reused across all 3 fine-tune notebooks so you only upload once.

Check the `/kaggle/input` listing above and fill in both TODO paths below before running.

In [5]:
import shutil
from pathlib import Path

# TODO: verify against the /kaggle/input listing cell above before running.
PROCESSED_SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in PROCESSED_SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


In [6]:
# TODO: verify against the /kaggle/input listing cell above before running -- this is the
# NEW checkpoint dataset (see markdown above), not the same as PROCESSED_SRC_DIR.
CHECKPOINT_SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/checkpoints")
CHECKPOINT_DST_DIR = Path("classification/new_way/Output/version1/checkpoints")
CHECKPOINT_DST_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
for item in CHECKPOINT_SRC_DIR.rglob("*.pth"):
    shutil.copy2(item, CHECKPOINT_DST_DIR / item.name)
    copied += 1
    print(f"  copied {item.name} ({item.stat().st_size / 1e6:.1f} MB)")
print(f"\n{copied} checkpoint file(s) staged to {CHECKPOINT_DST_DIR}")

  copied best_convnext_base_palpebral_new_way.pth (350.4 MB)
  copied best_coatnet_3_palpebral_new_way.pth (654.9 MB)
  copied best_efficientnet_b3_forniceal_palpebral_new_way.pth (43.4 MB)

3 checkpoint file(s) staged to classification/new_way/Output/version1/checkpoints


In [7]:
# Fails loudly here, not deep inside training, if either data source above is missing/misconfigured.
manifest_path = Path("classification/new_way/Offline_data_augmentation/manifest.csv")
assert manifest_path.exists(), (
    f"{manifest_path} not found -- Offline_data_augmentation/ should have arrived via git clone. "
    "Was it actually committed and pushed?"
)

checkpoint_path = CHECKPOINT_DST_DIR / "best_coatnet_3_palpebral_new_way.pth"
assert checkpoint_path.exists(), (
    f"{checkpoint_path} not found -- fix CHECKPOINT_SRC_DIR above to point at the real "
    "checkpoint-dataset mount path (check the /kaggle/input listing cell)."
)
print("Both data sources present:")
print(f"  {manifest_path}")
print(f"  {checkpoint_path} ({checkpoint_path.stat().st_size / 1e6:.1f} MB)")

Both data sources present:
  classification/new_way/Offline_data_augmentation/manifest.csv
  classification/new_way/Output/version1/checkpoints/best_coatnet_3_palpebral_new_way.pth (654.9 MB)


## Sanity check

Builds the fine-tune model for real (loads the checkpoint, applies the freeze/unfreeze split) and
confirms the trainable-parameter count matches what was verified locally, before any real
training starts.

In [8]:
import sys
sys.path.insert(0, "classification/new_way/Fine_tune")

import finetune_engine_coatnet3 as fe

model = fe.build_finetune_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}")
assert f"{trainable:,}" == "9,457,585", (
    f"Expected 9,457,585 trainable params, got {trainable:,} -- "
    "something about the checkpoint or freeze/unfreeze logic doesn't match the local verification."
)
print("Matches the locally-verified trainable-parameter count.")
del model

model.safetensors:   0%|          | 0.00/727M [00:00<?, ?B/s]

Trainable params: 9,457,585 / 163,637,965
Matches the locally-verified trainable-parameter count.


## Output syncing

In [9]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate Fine_tune/Output/{checkpoints,logs,plots}/ into a single top-level
    /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/coatnet_3_finetune_results.zip."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/new_way/Fine_tune/Output") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/coatnet_3_finetune_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

[sync_outputs] 35 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/coatnet_3_finetune_results.zip


## Training

Reproduces the exact, already-settled local configuration for CoAtNet-3 -- fixed
hyperparameters (no Optuna), discriminative LRs, val-F1-tracked scheduler/early-stopping. Prints
the baseline (pre-fine-tune) metrics first, then trains, then prints a before/after comparison on
the sealed test set.

In [10]:
!python classification/new_way/Fine_tune/train_finetune_coatnet3_palpebral.py
sync_outputs()

Using device: cuda
Model: coatnet_3_palpebral_new_way_finetune_attn
Source checkpoint: /kaggle/working/eyes-defy-anemia/classification/new_way/Output/version1/checkpoints/best_coatnet_3_palpebral_new_way.pth
train n=230 val n=33 test n=33
Trainable params: 9,457,585 / 163,637,965

--- Baseline (frozen-backbone checkpoint, before fine-tuning) ---
baseline val_f1=0.8966 val_auc=0.9661654135338347
baseline test_f1=0.8125 test_auc=0.9135338345864662
baseline India AUC=0.5909090909090908 Italy AUC=1.0

head_lr=1.125e-03  attn_lr=1.125e-04  weight_decay=1.186e-04
[coatnet_3_palpebral_new_way_finetune_attn] New best val_f1=0.7200 -> saved coatnet_3_palpebral_new_way_finetune_attn_best.pth
[coatnet_3_palpebral_new_way_finetune_attn] Epoch  1/60 - train_loss=0.8903 val_loss=0.5277 val_f1=0.7200 head_lr=1.12e-03 attn_lr=1.12e-04
[coatnet_3_palpebral_new_way_finetune_attn] New best val_f1=0.8000 -> saved coatnet_3_palpebral_new_way_finetune_attn_best.pth
[coatnet_3_palpebral_new_way_finetune_attn

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` and zipped to
`/kaggle/working/coatnet_3_finetune_results.zip`. Both are visible in this notebook version's
**Output** tab once you Save Version -> Save & Run All.

The headline number is in `{model_name}_history.json`'s `best_val_f1` and `test_metrics` --
compare against this notebook's own printed before/after block, and against the local run
recorded in `classification/.project_memory/13_finetune_pilot_programme.md`.

In [11]:
print("Final contents of /kaggle/working/outputs:")
for f in sorted(Path("/kaggle/working/outputs").rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to('/kaggle/working/outputs')}  ({f.stat().st_size / 1e6:.2f} MB)")

zip_path = Path("/kaggle/working/coatnet_3_finetune_results.zip")
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

Final contents of /kaggle/working/outputs:
  checkpoints/coatnet_3_palpebral_new_way_finetune_attn_best.pth  (654.92 MB)
  logs/coatnet_3_palpebral_new_way_finetune_attn_history.json  (0.01 MB)
  logs/convnext_base_palpebral_new_way_finetune_block3_history.json  (0.02 MB)
  logs/convnext_base_palpebral_new_way_finetune_block3_v2_history.json  (0.01 MB)
  logs/efficientnet_b3_forniceal_palpebral_new_way_finetune_block_history.json  (0.01 MB)
  logs/efficientnet_b3_forniceal_palpebral_new_way_finetune_block_v2_history.json  (0.01 MB)
  plots/coatnet_3_palpebral_new_way_finetune_attn_before_after.png  (0.05 MB)
  plots/coatnet_3_palpebral_new_way_finetune_attn_confusion_matrices.png  (0.05 MB)
  plots/coatnet_3_palpebral_new_way_finetune_attn_loss_curve.png  (0.08 MB)
  plots/coatnet_3_palpebral_new_way_finetune_attn_lr_schedule.png  (0.04 MB)
  plots/coatnet_3_palpebral_new_way_finetune_attn_roc_curves.png  (0.09 MB)
  plots/coatnet_3_palpebral_new_way_finetune_attn_val_metrics_curve.png